In [96]:
import os
import sys
import subprocess
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split

REPO = 'projeto-itbi-cin0208'

if not os.path.exists('src/carregar_dados.py'):
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
    elif not os.path.isdir(REPO):
        subprocess.run(
            ['git', 'clone', '-b', 'main',
             f'https://github.com/CaioC11/{REPO}.git'],
            check=True
        )
        os.chdir(REPO)
    else:
        os.chdir(REPO)

sys.path.insert(0, os.getcwd())

from src.carregar_dados import carregar_dados

print('Pasta atual:', os.getcwd())

Pasta atual: /content/projeto-itbi-cin0208


In [97]:
df = carregar_dados()

colunas_categoricas = [
    'bairro',
    'tipo_imovel',
    'tipo_construcao',
    'padrao_acabamento',
    'tipo_ocupacao',
    'estado_conservacao'
]

colunas_necessarias = colunas_categoricas + ['valor_avaliacao']
ausentes = [col for col in colunas_necessarias if col not in df.columns]
if ausentes:
    raise ValueError(f'Colunas necessárias ausentes no dataset: {ausentes}')

resumo = pd.DataFrame({
    'categorias': df[colunas_categoricas].nunique(),
    'nulos': df[colunas_categoricas].isna().sum(),
    'categorias raras (<30)': [
        (df[col].value_counts() < 30).sum()
        for col in colunas_categoricas
    ]
})

display(resumo)

,categorias,nulos,categorias raras (<30)
bairro,87,0,33
tipo_imovel,18,0,10
tipo_construcao,16,0,6
padrao_acabamento,3,0,0
tipo_ocupacao,3,0,0
estado_conservacao,3,0,1


In [98]:
class OneHotRaras(BaseEstimator, TransformerMixin):
    """One-Hot Encoding com categorias aprendidas somente no fit."""

    def __init__(self, colunas, min_count=None):
        self.colunas = colunas
        self.min_count = min_count

    def fit(self, X, y=None):
        self.categorias_ = {}
        self.agrupa_ = {}

        for col in self.colunas:
            contagem = X[col].value_counts(dropna=False)

            if self.min_count is None:
                manter = contagem
            else:
                manter = contagem[contagem >= self.min_count]

            self.categorias_[col] = list(manter.index)
            self.agrupa_[col] = len(manter) < len(contagem)

        return self

    def transform(self, X):
        X = X.copy()
        novas_colunas = {}

        for col in self.colunas:
            serie = X[col]
            categorias = self.categorias_[col]

            if self.agrupa_[col]:
                serie = serie.where(serie.isin(categorias), 'outras')
                categorias = categorias + ['outras']

            for categoria in categorias:
                nome = f'{col}_{categoria}'
                novas_colunas[nome] = (serie == categoria).astype(int)

        dummies = pd.DataFrame(novas_colunas, index=X.index)
        return pd.concat([X.drop(columns=self.colunas), dummies], axis=1)


class OrdinalOrdenado(BaseEstimator, TransformerMixin):
    """Codifica categorias segundo uma ordem definida manualmente."""

    def __init__(self, ordens):
        self.ordens = ordens

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        for col, ordem in self.ordens.items():
            mapa = {categoria: i + 1 for i, categoria in enumerate(ordem)}
            X[col] = X[col].map(mapa).fillna(0).astype(int)

        return X


class FrequencyEncoding(BaseEstimator, TransformerMixin):
    """Substitui cada categoria pela sua frequência relativa no treino."""

    def __init__(self, colunas):
        self.colunas = colunas

    def fit(self, X, y=None):
        self.freq_ = {
            col: X[col].value_counts(normalize=True)
            for col in self.colunas
        }
        return self

    def transform(self, X):
        X = X.copy()

        for col in self.colunas:
            X[col] = X[col].map(self.freq_[col]).fillna(0.0)

        return X


class TargetEncodingSuave(BaseEstimator, TransformerMixin):
    """Target Encoding com suavização em direção à média global do treino."""

    def __init__(self, colunas, suavizacao=10):
        self.colunas = colunas
        self.suavizacao = suavizacao

    def fit(self, X, y):
        y = pd.Series(np.asarray(y), index=X.index)

        self.media_global_ = y.mean()
        self.mapas_ = {}

        for col in self.colunas:
            grupos = y.groupby(X[col]).agg(['mean', 'count'])

            mapa = (
                grupos['count'] * grupos['mean']
                + self.suavizacao * self.media_global_
            ) / (grupos['count'] + self.suavizacao)

            self.mapas_[col] = mapa

        return self

    def transform(self, X):
        X = X.copy()

        for col in self.colunas:
            X[col] = X[col].map(self.mapas_[col]).fillna(self.media_global_)

        return X


class TargetEncodingForte(TargetEncodingSuave):
    """Target Encoding com suavização mais forte (padrão: 100)."""

    def __init__(self, colunas, suavizacao=100):
        super().__init__(colunas, suavizacao)

class TratamentoSFH(BaseEstimator, TransformerMixin):
    """Aplica o tratamento na coluna 'sfh': 'remover' ou 'binarizar' (sfh > 0)."""

    def __init__(self, modo='binarizar', coluna='sfh'):
        if modo not in ('remover', 'binarizar'):
            raise ValueError("O modo deve ser 'remover' ou 'binarizar'.")
        self.modo = modo
        self.coluna = coluna

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if self.coluna not in X.columns:
            raise ValueError(f"Coluna '{self.coluna}' não encontrada no DataFrame.")

        if self.modo == 'remover':
            return X.drop(columns=[self.coluna])
        elif self.modo == 'binarizar':
            X[self.coluna] = (X[self.coluna].fillna(0) > 0).astype(int)
            return X

class OneHotPesoReduzido(BaseEstimator, TransformerMixin):
    """One-Hot Encoding onde os valores ativos recebem um peso reduzido (< 1.0)."""

    def __init__(self, colunas, peso=0.5, min_count=None):
        self.colunas = colunas
        self.peso = peso
        self.min_count = min_count

    def fit(self, X, y=None):
        self.categorias_ = {}
        self.agrupa_ = {}

        for col in self.colunas:
            contagem = X[col].value_counts(dropna=False)
            if self.min_count is None:
                manter = contagem
            else:
                manter = contagem[contagem >= self.min_count]

            self.categorias_[col] = list(manter.index)
            self.agrupa_[col] = len(manter) < len(contagem)

        return self

    def transform(self, X):
        X = X.copy()
        novas_colunas = {}

        for col in self.colunas:
            serie = X[col]
            categorias = self.categorias_[col]

            if self.agrupa_[col]:
                serie = serie.where(serie.isin(categorias), 'outras')
                categorias = categorias + ['outras']

            for categoria in categorias:
                nome = f'{col}_{categoria}'
                novas_colunas[nome] = (serie == categoria).astype(float) * self.peso

        dummies = pd.DataFrame(novas_colunas, index=X.index)
        return pd.concat([X.drop(columns=self.colunas), dummies], axis=1)


class TratamentoFaltantes(BaseEstimator, TransformerMixin):

    def __init__(self, estrategia='mediana_bairro'):
        estrategias_validas = (
            'remover_linhas',
            'mediana_bairro',
            'mediana_bairro_indicador',
            'sem_lat_lon',
            'area_terreno_tipo',
            'media'
        )
        if estrategia not in estrategias_validas:
            raise ValueError(f'Estratégia inválida: {estrategia}')
        self.estrategia = estrategia

    def fit(self, X, y=None):
        self.cols_numericas_ = X.select_dtypes(include=[np.number]).columns.tolist()
        self.cols_com_nulos_ = [c for c in self.cols_numericas_ if X[c].isna().any()]
        self.mediana_global_ = X[self.cols_numericas_].median()
        self.media_global_ = X[self.cols_numericas_].mean()

        if 'bairro' in X.columns:
            self.mediana_por_bairro_ = X.groupby('bairro')[self.cols_numericas_].median()
        else:
            self.mediana_por_bairro_ = None

        if 'tipo_imovel' in X.columns and 'area_terreno' in X.columns:
            self.mediana_terreno_por_tipo_ = X.groupby('tipo_imovel')['area_terreno'].median()
        else:
            self.mediana_terreno_por_tipo_ = None

        self.cols_lat_lon_ = [
            c for c in X.columns if 'lat' in c.lower() or 'lon' in c.lower()
        ]
        return self

    def transform(self, X):
        X = X.copy()

        if self.estrategia == 'remover_linhas':
            return X.dropna()

        if self.estrategia == 'media':
            for col in self.cols_numericas_:
                if col in X.columns:
                    X[col] = X[col].fillna(self.media_global_[col])
            return X

        if self.estrategia == 'sem_lat_lon':
            X = X.drop(columns=[c for c in self.cols_lat_lon_ if c in X.columns])
            for col in self.cols_numericas_:
                if col in X.columns:
                    X[col] = X[col].fillna(self.mediana_global_[col])
            return X

        if self.estrategia == 'area_terreno_tipo':
            if 'area_terreno' in X.columns and self.mediana_terreno_por_tipo_ is not None and 'tipo_imovel' in X.columns:
                mapa_terreno = X['tipo_imovel'].map(self.mediana_terreno_por_tipo_)
                X['area_terreno'] = X['area_terreno'].fillna(mapa_terreno).fillna(self.mediana_global_['area_terreno'])
            for col in self.cols_numericas_:
                if col in X.columns and col != 'area_terreno':
                    X[col] = X[col].fillna(self.mediana_global_[col])
            return X

        if self.estrategia in ('mediana_bairro', 'mediana_bairro_indicador'):
            if self.estrategia == 'mediana_bairro_indicador':
                for col in self.cols_com_nulos_:
                    if col in X.columns:
                        X[f'{col}_missing'] = X[col].isna().astype(int)

            for col in self.cols_numericas_:
                if col not in X.columns:
                    continue
                if self.mediana_por_bairro_ is not None and 'bairro' in X.columns:
                    mapa_bairro = X['bairro'].map(self.mediana_por_bairro_[col])
                    X[col] = X[col].fillna(mapa_bairro)
                X[col] = X[col].fillna(self.mediana_global_[col])
            return X


class WinsorizadorOutliers(BaseEstimator, TransformerMixin):

    def __init__(self, q_inf=0.01, q_sup=0.99, colunas=None):
        self.q_inf = q_inf
        self.q_sup = q_sup
        self.colunas = colunas

    def fit(self, X, y=None):
        if self.colunas is None:
            # Aplica apenas em colunas numéricas contínuas (exclui binárias 0/1)
            self.colunas_alvo_ = [
                c for c in X.select_dtypes(include=[np.number]).columns
                if X[c].nunique(dropna=True) > 2
            ]
        else:
            self.colunas_alvo_ = [c for c in self.colunas if c in X.columns]

        self.limites_ = {}
        for col in self.colunas_alvo_:
            low = X[col].quantile(self.q_inf)
            high = X[col].quantile(self.q_sup)
            self.limites_[col] = (low, high)
        return self

    def transform(self, X):
        X = X.copy()
        for col, (low, high) in self.limites_.items():
            X[col] = X[col].clip(lower=low, upper=high)
        return X


def filtro_treino_iqr(X_train, y_train, multiplicador=1.5):
    q1 = y_train.quantile(0.25)
    q3 = y_train.quantile(0.75)
    iqr = q3 - q1
    lim_inf = q1 - multiplicador * iqr
    lim_sup = q3 + multiplicador * iqr

    mascara = (y_train >= lim_inf) & (y_train <= lim_sup)
    return X_train.loc[mascara].copy(), y_train.loc[mascara].copy()

In [99]:
y = np.log1p(df['valor_avaliacao'])
X = df.drop(columns=['valor_avaliacao']).copy()

X_train_base, X_test_base, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

assert set(X_train_base.index).isdisjoint(set(X_test_base.index))
assert X_train_base.index.equals(y_train.index)
assert X_test_base.index.equals(y_test.index)

print('Treino:', X_train_base.shape)
print('Teste:', X_test_base.shape)
print('Índices de treino e teste separados corretamente.')

Treino: (13054, 21)
Teste: (3264, 21)
Índices de treino e teste separados corretamente.


In [100]:
ordens_ordinal = {
    'padrao_acabamento': ['Simples', 'Médio', 'Superior'],
    'estado_conservacao': ['Mau', 'Regular', 'Bom']
}

ordem_acabamento = {
    'padrao_acabamento': ['Simples', 'Médio', 'Superior']
}

colunas_alta_cardinalidade = [
    'bairro',
    'tipo_imovel',
    'tipo_construcao'
]

colunas_baixa_cardinalidade = [
    'tipo_ocupacao',
    'estado_conservacao'
]

colunas_nominais_ordinal = [
    'bairro',
    'tipo_imovel',
    'tipo_construcao',
    'tipo_ocupacao'
]

In [101]:
ordinal = OrdinalOrdenado(ordens_ordinal).fit(X_train_base)

X_train_ordinal = ordinal.transform(X_train_base)
X_test_ordinal = ordinal.transform(X_test_base)

onehot_ordinal = OneHotRaras(colunas_nominais_ordinal).fit(X_train_ordinal)

e1_train = onehot_ordinal.transform(X_train_ordinal)
e1_test = onehot_ordinal.transform(X_test_ordinal)

print('Ordinal — treino:', e1_train.shape)
print('Ordinal — teste:', e1_test.shape)

Ordinal — treino: (13054, 141)
Ordinal — teste: (3264, 141)


In [102]:
ordinal_comum = OrdinalOrdenado(ordem_acabamento).fit(X_train_base)

X_train_ord = ordinal_comum.transform(X_train_base)
X_test_ord = ordinal_comum.transform(X_test_base)

In [103]:
frequency_encoder = FrequencyEncoding(colunas_alta_cardinalidade)
frequency_encoder.fit(X_train_ord)

frequency_train = frequency_encoder.transform(X_train_ord)
frequency_test = frequency_encoder.transform(X_test_ord)

onehot_frequency = OneHotRaras(colunas_baixa_cardinalidade)
onehot_frequency.fit(frequency_train)

e2_train = onehot_frequency.transform(frequency_train)
e2_test = onehot_frequency.transform(frequency_test)

print('Frequency — treino:', e2_train.shape)
print('Frequency — teste:', e2_test.shape)

Frequency — treino: (13054, 25)
Frequency — teste: (3264, 25)


In [104]:
target_suave = TargetEncodingSuave(
    colunas_alta_cardinalidade,
    suavizacao=10
)
target_suave.fit(X_train_ord, y_train)

target_suave_train = target_suave.transform(X_train_ord)
target_suave_test = target_suave.transform(X_test_ord)

onehot_target_suave = OneHotRaras(colunas_baixa_cardinalidade)
onehot_target_suave.fit(target_suave_train)

e3_train = onehot_target_suave.transform(target_suave_train)
e3_test = onehot_target_suave.transform(target_suave_test)

print('Target suave — treino:', e3_train.shape)
print('Target suave — teste:', e3_test.shape)

Target suave — treino: (13054, 25)
Target suave — teste: (3264, 25)


In [105]:
target_forte = TargetEncodingForte(
    colunas_alta_cardinalidade,
    suavizacao=100
)
target_forte.fit(X_train_ord, y_train)

target_forte_train = target_forte.transform(X_train_ord)
target_forte_test = target_forte.transform(X_test_ord)

onehot_target_forte = OneHotRaras(colunas_baixa_cardinalidade)
onehot_target_forte.fit(target_forte_train)

e4_train = onehot_target_forte.transform(target_forte_train)
e4_test = onehot_target_forte.transform(target_forte_test)

print('Target forte — treino:', e4_train.shape)
print('Target forte — teste:', e4_test.shape)

Target forte — treino: (13054, 25)
Target forte — teste: (3264, 25)


In [106]:
experimentos = {
    'ordinal': (e1_train, e1_test),
    'frequency': (e2_train, e2_test),
    'target_suave': (e3_train, e3_test),
    'target_forte': (e4_train, e4_test)
}

for nome, (treino, teste) in experimentos.items():
    assert list(treino.columns) == list(teste.columns), (
        f'Colunas diferentes em {nome}'
    )

print('Todas as variantes possuem colunas consistentes!')

assert np.isclose(target_suave.media_global_, y_train.mean())
assert np.isclose(target_forte.media_global_, y_train.mean())

print('Target suave e forte ajustados com a média do treino!')

mapa_acabamento = {
    'Simples': 1,
    'Médio': 2,
    'Superior': 3
}

esperado_train = X_train_base['padrao_acabamento'].map(mapa_acabamento).fillna(0).astype(int)
esperado_test = X_test_base['padrao_acabamento'].map(mapa_acabamento).fillna(0).astype(int)

for nome, (treino, teste) in experimentos.items():
    pd.testing.assert_series_equal(
        treino['padrao_acabamento'],
        esperado_train,
        check_names=False
    )
    pd.testing.assert_series_equal(
        teste['padrao_acabamento'],
        esperado_test,
        check_names=False
    )

print('padrao_acabamento está ordinal em todas as variantes!')

novo = X_test_base.head(1).copy()
novo['padrao_acabamento'] = 'ACABAMENTO_INEXISTENTE'
novo['bairro'] = 'BAIRRO_INEXISTENTE'

novo_ordinal = ordinal_comum.transform(novo)
assert novo_ordinal['padrao_acabamento'].iloc[0] == 0

assert np.isclose(
    target_suave.transform(novo)['bairro'].iloc[0],
    target_suave.media_global_
)
assert np.isclose(
    target_forte.transform(novo)['bairro'].iloc[0],
    target_forte.media_global_
)

assert frequency_encoder.transform(novo)['bairro'].iloc[0] == 0.0

print('Categorias desconhecidas tratadas corretamente!')
print('Treino e teste permanecem separados!')

Todas as variantes possuem colunas consistentes!
Target suave e forte ajustados com a média do treino!
padrao_acabamento está ordinal em todas as variantes!
Categorias desconhecidas tratadas corretamente!
Treino e teste permanecem separados!


In [107]:
resumo_experimentos = pd.DataFrame([
    {
        'experimento': nome,
        'linhas_treino': treino.shape[0],
        'colunas_treino': treino.shape[1],
        'linhas_teste': teste.shape[0],
        'colunas_teste': teste.shape[1]
    }
    for nome, (treino, teste) in experimentos.items()
])

display(resumo_experimentos)

,experimento,linhas_treino,colunas_treino,linhas_teste,colunas_teste
0,ordinal,13054,141,3264,141
1,frequency,13054,25,3264,25
2,target_suave,13054,25,3264,25
3,target_forte,13054,25,3264,25


In [108]:
X_base_tr, X_base_te = e3_train, e3_test

sfh_remover = TratamentoSFH(modo='remover').fit(X_base_tr)
t39_sem_sfh_train = sfh_remover.transform(X_base_tr)
t39_sem_sfh_test = sfh_remover.transform(X_base_te)

sfh_binarizar = TratamentoSFH(modo='binarizar').fit(X_base_tr)
t39_bin_sfh_train = sfh_binarizar.transform(X_base_tr)
t39_bin_sfh_test = sfh_binarizar.transform(X_base_te)

experimentos_t39 = {
    't39_sem_sfh': (t39_sem_sfh_train, t39_sem_sfh_test),
    't39_sfh_binarizado': (t39_bin_sfh_train, t39_bin_sfh_test)
}

assert 'sfh' not in t39_sem_sfh_train.columns
assert 'sfh' not in t39_sem_sfh_test.columns
assert set(t39_bin_sfh_train['sfh'].unique()).issubset({0, 1})
assert set(t39_bin_sfh_test['sfh'].unique()).issubset({0, 1})
assert t39_sem_sfh_train.shape[1] == X_base_tr.shape[1] - 1
assert t39_bin_sfh_train.shape[1] == X_base_tr.shape[1]

print('Task 39 — Variantes geradas e validadas com sucesso!')
for nome, (tr, te) in experimentos_t39.items():
    print(f'{nome}: treino={tr.shape}, teste={te.shape}')

Task 39 — Variantes geradas e validadas com sucesso!
t39_sem_sfh: treino=(13054, 24), teste=(3264, 24)
t39_sfh_binarizado: treino=(13054, 25), teste=(3264, 25)


In [110]:
te_bairro = TargetEncodingSuave(['bairro'], suavizacao=10).fit(X_train_ord, y_train)
hib_tr = te_bairro.transform(X_train_ord)
hib_te = te_bairro.transform(X_test_ord)

colunas_resto_onehot = ['tipo_imovel', 'tipo_construcao', 'tipo_ocupacao', 'estado_conservacao']
oh_resto = OneHotRaras(colunas_resto_onehot).fit(hib_tr)
f2_2_hibrido_train = oh_resto.transform(hib_tr)
f2_2_hibrido_test = oh_resto.transform(hib_te)

oh_bairro_raros = OneHotRaras(['bairro'], min_count=30).fit(X_train_ord)
br_tr = oh_bairro_raros.transform(X_train_ord)
br_te = oh_bairro_raros.transform(X_test_ord)

oh_demais = OneHotRaras(colunas_resto_onehot).fit(br_tr)
f2_2_bairro_raros_train = oh_demais.transform(br_tr)
f2_2_bairro_raros_test = oh_demais.transform(br_te)

sem_b_tr = X_train_ord.drop(columns=['bairro'])
sem_b_te = X_test_ord.drop(columns=['bairro'])

oh_sem_bairro = OneHotRaras(colunas_resto_onehot).fit(sem_b_tr)
f2_2_sem_bairro_train = oh_sem_bairro.transform(sem_b_tr)
f2_2_sem_bairro_test = oh_sem_bairro.transform(sem_b_te)

colunas_todas_onehot = ['bairro'] + colunas_resto_onehot
oh_peso_red = OneHotPesoReduzido(colunas_todas_onehot, peso=0.5).fit(X_train_ord)
f2_2_onehot_peso_train = oh_peso_red.transform(X_train_ord)
f2_2_onehot_peso_test = oh_peso_red.transform(X_test_ord)

experimentos_f2_2 = {
    'f2_2_hibrido': (f2_2_hibrido_train, f2_2_hibrido_test),
    'f2_2_bairro_raros_30': (f2_2_bairro_raros_train, f2_2_bairro_raros_test),
    'f2_2_sem_bairro': (f2_2_sem_bairro_train, f2_2_sem_bairro_test),
    'f2_2_onehot_peso_reduzido': (f2_2_onehot_peso_train, f2_2_onehot_peso_test)
}

for nome, (tr, te) in experimentos_f2_2.items():
    assert list(tr.columns) == list(te.columns), f'Colunas diferentes em {nome}'
    cols_verificacao = [
        c for c in tr.columns
        if c == 'bairro' or c.startswith(('bairro_', 'tipo_imovel_', 'tipo_construcao_', 'tipo_ocupacao_', 'estado_conservacao_'))
    ]
    assert tr[cols_verificacao].isna().sum().sum() == 0, f'Nulos encontrados no treino de {nome}'
    assert te[cols_verificacao].isna().sum().sum() == 0, f'Nulos encontrados no teste de {nome}'

assert 'bairro_outras' in f2_2_bairro_raros_train.columns
assert not any(c.startswith('bairro') for c in f2_2_sem_bairro_train.columns)
assert f2_2_onehot_peso_train.filter(like='bairro_').max().max() == 0.5

resumo_f2_2 = pd.DataFrame([
    {
        'experimento': nome,
        'linhas_treino': tr.shape[0],
        'n_colunas': tr.shape[1],
        'linhas_teste': te.shape[0]
    }
    for nome, (tr, te) in experimentos_f2_2.items()
])

print('F2-2 — Variantes geradas e validadas com sucesso!')
display(resumo_f2_2)

F2-2 — Variantes geradas e validadas com sucesso!


,experimento,linhas_treino,n_colunas,linhas_teste
0,f2_2_hibrido,13054,57,3264
1,f2_2_bairro_raros_30,13054,105,3264
2,f2_2_sem_bairro,13054,56,3264
3,f2_2_onehot_peso_reduzido,13054,143,3264


In [112]:
def aplicar_encoding_padrao(X_tr_raw, y_tr_raw, X_te_raw):
    ord_enc = OrdinalOrdenado(ordem_acabamento).fit(X_tr_raw)
    tr = ord_enc.transform(X_tr_raw)
    te = ord_enc.transform(X_te_raw)

    te_enc = TargetEncodingSuave(colunas_alta_cardinalidade, suavizacao=10).fit(tr, y_tr_raw)
    tr = te_enc.transform(tr)
    te = te_enc.transform(te)

    oh_enc = OneHotRaras(colunas_baixa_cardinalidade).fit(tr)
    return oh_enc.transform(tr), oh_enc.transform(te)

imp_rem = TratamentoFaltantes('remover_linhas').fit(X_train_base)
rem_tr_raw = imp_rem.transform(X_train_base)
rem_te_raw = imp_rem.transform(X_test_base)
y_train_sem_faltantes = y_train.loc[rem_tr_raw.index]
y_test_sem_faltantes = y_test.loc[rem_te_raw.index]

f1_2_rem_tr, f1_2_rem_te = aplicar_encoding_padrao(rem_tr_raw, y_train_sem_faltantes, rem_te_raw)

estrategias_imp = [
    'mediana_bairro',
    'mediana_bairro_indicador',
    'sem_lat_lon',
    'area_terreno_tipo',
    'media'
]

experimentos_f1_2 = {
    'f1_2_remover_linhas': (f1_2_rem_tr, y_train_sem_faltantes, f1_2_rem_te, y_test_sem_faltantes)
}

for est in estrategias_imp:
    imp = TratamentoFaltantes(est).fit(X_train_base)
    tr_imp = imp.transform(X_train_base)
    te_imp = imp.transform(X_test_base)
    tr_cod, te_cod = aplicar_encoding_padrao(tr_imp, y_train, te_imp)
    experimentos_f1_2[f'f1_2_{est}'] = (tr_cod, y_train, te_cod, y_test)

for nome, (tr, y_tr_exp, te, y_te_exp) in experimentos_f1_2.items():
    assert list(tr.columns) == list(te.columns), f'Colunas diferentes em {nome}'
    assert tr.index.equals(y_tr_exp.index), f'Índices desalinhados no treino de {nome}'
    assert te.index.equals(y_te_exp.index), f'Índices desalinhados no teste de {nome}'
    assert tr.select_dtypes(include=[np.number]).isna().sum().sum() == 0, f'Nulos numéricos no treino de {nome}'
    assert te.select_dtypes(include=[np.number]).isna().sum().sum() == 0, f'Nulos numéricos no teste de {nome}'

assert not any('lat' in c.lower() or 'lon' in c.lower() for c in experimentos_f1_2['f1_2_sem_lat_lon'][0].columns)
assert any(c.endswith('_missing') for c in experimentos_f1_2['f1_2_mediana_bairro_indicador'][0].columns)

resumo_f1_2 = pd.DataFrame([
    {
        'experimento': nome,
        'conjunto': 'sem_faltantes' if nome == 'f1_2_remover_linhas' else 'completo',
        'linhas_treino': tr.shape[0],
        'colunas_treino': tr.shape[1],
        'linhas_teste': te.shape[0]
    }
    for nome, (tr, _, te, _) in experimentos_f1_2.items()
])

print('Issue #37 (F1-2 Faltantes) — 6 variantes geradas e validadas!')
display(resumo_f1_2)

Issue #37 (F1-2 Faltantes) — 6 variantes geradas e validadas!


,experimento,conjunto,linhas_treino,colunas_treino,linhas_teste
0,f1_2_remover_linhas,sem_faltantes,7446,25,1870
1,f1_2_mediana_bairro,completo,13054,25,3264
2,f1_2_mediana_bairro_indicador,completo,13054,27,3264
3,f1_2_sem_lat_lon,completo,13054,23,3264
4,f1_2_area_terreno_tipo,completo,13054,25,3264
5,f1_2_media,completo,13054,25,3264


In [113]:
# Partindo do baseline codificado e3_train / e3_test (y_train e y_test já estão em log1p)
X_out_base_tr, X_out_base_te = e3_train.copy(), e3_test.copy()

# 1. Remoção por IQR no log (1.5) só no treino
f1_3_iqr_1_5_tr, y_train_iqr_1_5 = filtro_treino_iqr(X_out_base_tr, y_train, multiplicador=1.5)
f1_3_iqr_1_5_te = X_out_base_te.copy()

# 2. Remoção por IQR no log (3.0) só no treino
f1_3_iqr_3_0_tr, y_train_iqr_3_0 = filtro_treino_iqr(X_out_base_tr, y_train, multiplicador=3.0)
f1_3_iqr_3_0_te = X_out_base_te.copy()

# 3. Winsorização 1-99 (limites aprendidos no treino)
win_1_99 = WinsorizadorOutliers(q_inf=0.01, q_sup=0.99).fit(X_out_base_tr)
f1_3_win_1_99_tr = win_1_99.transform(X_out_base_tr)
f1_3_win_1_99_te = win_1_99.transform(X_out_base_te)

# 4. Winsorização 5-95 (limites aprendidos no treino)
win_5_95 = WinsorizadorOutliers(q_inf=0.05, q_sup=0.95).fit(X_out_base_tr)
f1_3_win_5_95_tr = win_5_95.transform(X_out_base_tr)
f1_3_win_5_95_te = win_5_95.transform(X_out_base_te)

experimentos_f1_3 = {
    'f1_3_iqr_log_1_5': (f1_3_iqr_1_5_tr, y_train_iqr_1_5, f1_3_iqr_1_5_te, y_test),
    'f1_3_iqr_log_3_0': (f1_3_iqr_3_0_tr, y_train_iqr_3_0, f1_3_iqr_3_0_te, y_test),
    'f1_3_winsor_1_99': (f1_3_win_1_99_tr, y_train, f1_3_win_1_99_te, y_test),
    'f1_3_winsor_5_95': (f1_3_win_5_95_tr, y_train, f1_3_win_5_95_te, y_test)
}

# Validações da Issue #38 (garantindo que o teste permanece intacto)
for nome, (tr, y_tr_exp, te, y_te_exp) in experimentos_f1_3.items():
    assert list(tr.columns) == list(te.columns), f'Colunas diferentes em {nome}'
    assert tr.index.equals(y_tr_exp.index), f'Índices desalinhados no treino de {nome}'
    assert te.shape == X_out_base_te.shape, f'O tamanho do teste foi alterado em {nome}!'
    assert te.index.equals(X_out_base_te.index), f'Índices do teste alterados em {nome}!'
    assert y_te_exp.equals(y_test), f'Alvo de teste alterado em {nome}!'

assert len(f1_3_iqr_1_5_tr) <= len(f1_3_iqr_3_0_tr) <= len(X_out_base_tr)

resumo_f1_3 = pd.DataFrame([
    {
        'experimento': nome,
        'linhas_treino': tr.shape[0],
        'removidas_treino': len(X_out_base_tr) - tr.shape[0],
        'colunas_treino': tr.shape[1],
        'linhas_teste_intacto': te.shape[0]
    }
    for nome, (tr, _, te, _) in experimentos_f1_3.items()
])

print('Issue #38 (F1-3 Outliers) — 4 variantes geradas e validadas!')
display(resumo_f1_3)

Issue #38 (F1-3 Outliers) — 4 variantes geradas e validadas!


,experimento,linhas_treino,removidas_treino,colunas_treino,linhas_teste_intacto
0,f1_3_iqr_log_1_5,12454,600,25,3264
1,f1_3_iqr_log_3_0,12966,88,25,3264
2,f1_3_winsor_1_99,13054,0,25,3264
3,f1_3_winsor_5_95,13054,0,25,3264


In [116]:
%%writefile src/limpeza.py
"""Pré-tratamento fixo e padronização de texto do dataset ITBI (Projeto ITBI, CIN0208)."""
import unicodedata

import numpy as np

from src.carregar_dados import carregar_dados

ALVO = "valor_avaliacao"


def _remover_acentos(texto):
    """Remove diacríticos (acentos/cedilha) mantendo apenas caracteres ASCII."""
    normalizado = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in normalizado if not unicodedata.combining(c))


def padronizar_bairros(serie):
    """Padroniza grafias de bairro: remove espaços extras, unifica acentuação e aplica Title Case."""
    s = serie.astype("string").str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.map(lambda x: _remover_acentos(x) if isinstance(x, str) else x)
    s = s.str.title()
    return s.astype(object)


def limpar_dados(df):
    """Aplica o pré-tratamento fixo do plano de experimentos sobre o DataFrame bruto."""
    df = df.drop_duplicates()  # 301 linhas exatamente duplicadas
    df = df[df[ALVO] > 100].copy()  # Remove o registro de R$ 19,52
    df.loc[df["area_terreno"] == 0, "area_terreno"] = np.nan
    df.loc[df["fracao_ideal"] == 0, "fracao_ideal"] = np.nan
    if "bairro" in df.columns:
        df["bairro"] = padronizar_bairros(df["bairro"])
    return df


def carregar_e_limpar(caminho=None):
    """Lê o CSV com o carregador do repositório e aplica o pré-tratamento fixo."""
    df = carregar_dados(caminho=caminho)
    return limpar_dados(df)

Overwriting src/limpeza.py


In [117]:
%%writefile src/baseline.py

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold, cross_validate
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

RAIZ = Path(__file__).resolve().parents[1]
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from src.limpeza import ALVO, carregar_e_limpar

SEED = 42
SAIDA = RAIZ / "resultados" / "baseline.csv"

NUMERICAS = ["ano_construcao", "area_terreno", "area_construida",
             "fracao_ideal", "sfh", "latitude", "longitude"]
CATEGORICAS = ["bairro", "padrao_acabamento", "tipo_construcao",
               "tipo_ocupacao", "estado_conservacao", "tipo_imovel"]


def pre_processamento(escalar):
    passos = [("imputacao", SimpleImputer(strategy="median"))]
    if escalar:
        passos.append(("escala", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(passos), NUMERICAS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS),
    ])


def modelo(regressor, escalar):
    pipe = Pipeline([("prep", pre_processamento(escalar)), ("reg", regressor)])
    return TransformedTargetRegressor(regressor=pipe, func=np.log1p,
                                      inverse_func=np.expm1)


def main():
    df = carregar_e_limpar()
    X, y = df[NUMERICAS + CATEGORICAS], df[ALVO]
    print(f"Instâncias após a limpeza: {len(df)} | scikit-learn {sklearn.__version__}")

    cv = KFold(n_splits=10, shuffle=True, random_state=SEED)
    scoring = {"mae": "neg_mean_absolute_error",
               "rmse": "neg_root_mean_squared_error",
               "mape": "neg_mean_absolute_percentage_error",
               "r2": "r2"}
    candidatos = {
        "Média (em log)": modelo(DummyRegressor(), False),
        "kNN k=5 sem escala": modelo(KNeighborsRegressor(), False),
        "kNN k=5 com escala": modelo(KNeighborsRegressor(), True),
    }

    linhas = []
    for nome, m in candidatos.items():
        r = cross_validate(m, X, y, cv=cv, scoring=scoring)
        linha = {"modelo": nome}
        for k in scoring:
            v = r[f"test_{k}"]
            sinal = 1 if k == "r2" else -1
            linha[f"{k}_media"] = sinal * v.mean()
            linha[f"{k}_desvio"] = v.std()
        linhas.append(linha)

    tabela = pd.DataFrame(linhas)
    SAIDA.parent.mkdir(exist_ok=True)
    tabela.round(3).to_csv(SAIDA, index=False)
    print(tabela.round(3).to_string(index=False))
    print(f"\nSalvo em {SAIDA}")


if __name__ == "__main__":
    main()

Overwriting src/baseline.py


In [118]:
from src.limpeza import carregar_e_limpar

df_limpo = carregar_e_limpar()

assert df_limpo.duplicated().sum() == 0
assert (df_limpo["valor_avaliacao"] <= 100).sum() == 0
assert (df_limpo["area_terreno"] == 0).sum() == 0
assert (df_limpo["fracao_ideal"] == 0).sum() == 0

bairros_limpos = sorted(df_limpo["bairro"].dropna().unique().tolist())
assert "Gracas" in bairros_limpos and "Graças" not in bairros_limpos
assert "Santo Antonio" in bairros_limpos and "Santo Antônio" not in bairros_limpos
assert "Sao Jose" in bairros_limpos and "São José" not in bairros_limpos
assert "Cohab Ibura" in bairros_limpos and "Cohab   Ibura" not in bairros_limpos

print(f"Total de bairros únicos após padronização: {len(bairros_limpos)}")
print("Limpeza fixa e padronização validadas com sucesso!\n")

!python src/baseline.py

Total de bairros únicos após padronização: 84
Limpeza fixa e padronização validadas com sucesso!

Instâncias após a limpeza: 16016 | scikit-learn 1.6.1
            modelo  mae_media  mae_desvio  rmse_media  rmse_desvio  mape_media  mape_desvio  r2_media  r2_desvio
    Média (em log) 414136.389   42036.870 2300199.367   870618.424       0.747        0.050    -0.016      0.008
kNN k=5 sem escala 200910.554   30373.798 1541696.269   708364.442       0.280        0.038     0.556      0.123
kNN k=5 com escala 173421.356   22939.060 1063816.927   291176.346       0.253        0.033     0.741      0.133

Salvo em /content/projeto-itbi-cin0208/resultados/baseline.csv


In [120]:
import sys
sys.path.insert(0, 'src')

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from src.executor import executar_experimento, SAIDA

# Garante que os resultados anteriores de teste local não dupliquem chaves caso rode mais de uma vez
if SAIDA.exists():
    SAIDA.unlink()

# Colunas base conforme baseline.py / executor.py
COLS_NUM_BASE = ["ano_construcao", "area_terreno", "area_construida", "fracao_ideal", "sfh", "latitude", "longitude"]
COLS_CAT_BASE = ["bairro", "padrao_acabamento", "tipo_construcao", "tipo_ocupacao", "estado_conservacao", "tipo_imovel"]

# Transformador auxiliar para imputar numéricas restantes (como B0 faz com median) e escalar tudo no final
class ImputarEEscalarFinal(BaseEstimator, TransformerMixin):
    def __init__(self, estrategia_num='median', escalar=True):
        self.estrategia_num = estrategia_num
        self.escalar = escalar

    def fit(self, X, y=None):
        self.colunas_ = list(X.columns)
        self.imputer_ = SimpleImputer(strategy=self.estrategia_num).fit(X)
        X_imp = self.imputer_.transform(X)
        self.scaler_ = StandardScaler().fit(X_imp) if self.escalar else None
        return self

    def transform(self, X):
        X_imp = self.imputer_.transform(X[self.colunas_])
        return self.scaler_.transform(X_imp) if self.scaler_ is not None else X_imp

def montar_pipeline(passos_customizados, estrategia_num='median'):
    """Monta um Pipeline sklearn compatível com executor.py (fit_transform(X, y) -> numpy array escalado)."""
    passos = list(passos_customizados) + [("final", ImputarEEscalarFinal(estrategia_num=estrategia_num))]
    return Pipeline(passos)

# Configuração base equivalente ao B0 (para variar um fator por vez conforme o plano)
CONFIG_B0 = {
    "faltantes": "mediana_global",
    "outliers": "nenhum",
    "sfh": "original",
    "encoding": "onehot",
    "escala": "standard",
    "selecao": "nenhuma",
    "pca": "nenhum",
    "ponderacao": "nenhuma",
    "desbalanceamento": "nenhum",
}

def rodar_e_logar(nome_desc, config_override, passos, conjunto="completo", filtro_treino=None, estrategia_num="median"):
    cfg = {**CONFIG_B0, **config_override}
    pipe = montar_pipeline(passos, estrategia_num=estrategia_num)
    res = executar_experimento(
        configuracao=cfg,
        preprocessamento=pipe,
        conjunto=conjunto,
        filtro_treino=filtro_treino,
        salvar=True
    )
    print(f"[{nome_desc}] MAE={res['mae_media']:.2f} | MAPE={res['mape_media']:.4f} | R2={res['r2_media']:.4f} | cols={int(res['n_colunas'])}")
    return res

resultados_todas_tasks = []

print("=== 0. BASELINES B0 (completo e sem_faltantes) ===")
pipe_b0 = [
    ("ord", OrdinalOrdenado(ordem_acabamento)),
    ("oh", OneHotRaras(['bairro', 'tipo_imovel', 'tipo_construcao', 'tipo_ocupacao', 'estado_conservacao']))
]
resultados_todas_tasks.append(("B0_completo", rodar_e_logar("B0 (completo)", {}, pipe_b0, conjunto="completo")))
resultados_todas_tasks.append(("B0_sem_faltantes", rodar_e_logar("B0 (sem_faltantes)", {"faltantes": "remover_linhas"}, pipe_b0, conjunto="sem_faltantes")))

print("\n=== 1. ENCODING F2-1 (4 variantes) ===")
# Ordinal (padrao_acabamento e estado_conservacao)
resultados_todas_tasks.append(("f2_1_ordinal", rodar_e_logar(
    "F2-1 Ordinal", {"encoding": "ordinal"},
    [("ord", OrdinalOrdenado(ordens_ordinal)), ("oh", OneHotRaras(colunas_nominais_ordinal))]
)))
# Frequency
resultados_todas_tasks.append(("f2_1_frequency", rodar_e_logar(
    "F2-1 Frequency", {"encoding": "frequency"},
    [("ord", OrdinalOrdenado(ordem_acabamento)), ("freq", FrequencyEncoding(colunas_alta_cardinalidade)), ("oh", OneHotRaras(colunas_baixa_cardinalidade))]
)))
# Target suave (10)
resultados_todas_tasks.append(("f2_1_target_suave", rodar_e_logar(
    "F2-1 Target Suave (10)", {"encoding": "target_suave_10"},
    [("ord", OrdinalOrdenado(ordem_acabamento)), ("te", TargetEncodingSuave(colunas_alta_cardinalidade, 10)), ("oh", OneHotRaras(colunas_baixa_cardinalidade))]
)))
# Target forte (100)
resultados_todas_tasks.append(("f2_1_target_forte", rodar_e_logar(
    "F2-1 Target Forte (100)", {"encoding": "target_forte_100"},
    [("ord", OrdinalOrdenado(ordem_acabamento)), ("te", TargetEncodingForte(colunas_alta_cardinalidade, 100)), ("oh", OneHotRaras(colunas_baixa_cardinalidade))]
)))

print("\n=== 2. ISSUE #42 — F2-2 VARIANTES EXTRAS (4 variantes) ===")
cols_resto_oh = ['tipo_imovel', 'tipo_construcao', 'tipo_ocupacao', 'estado_conservacao']
# Híbrido
resultados_todas_tasks.append(("f2_2_hibrido", rodar_e_logar(
    "F2-2 Híbrido", {"encoding": "hibrido_te_bairro_oh_resto"},
    [("ord", OrdinalOrdenado(ordem_acabamento)), ("te_b", TargetEncodingSuave(['bairro'], 10)), ("oh", OneHotRaras(cols_resto_oh))]
)))
# Bairro raros < 30
resultados_todas_tasks.append(("f2_2_bairro_raros_30", rodar_e_logar(
    "F2-2 Bairro Raros (<30)", {"encoding": "onehot_bairro_raros_30"},
    [("ord", OrdinalOrdenado(ordem_acabamento)), ("oh_b", OneHotRaras(['bairro'], min_count=30)), ("oh", OneHotRaras(cols_resto_oh))]
)))
# Sem bairro
class DropColunas(BaseEstimator, TransformerMixin):
    def __init__(self, cols): self.cols = cols
    def fit(self, X, y=None): return self
    def transform(self, X): return X.drop(columns=[c for c in self.cols if c in X.columns])

resultados_todas_tasks.append(("f2_2_sem_bairro", rodar_e_logar(
    "F2-2 Sem Bairro", {"encoding": "onehot_sem_bairro"},
    [("drop_b", DropColunas(['bairro'])), ("ord", OrdinalOrdenado(ordem_acabamento)), ("oh", OneHotRaras(cols_resto_oh))]
)))
# One-Hot peso reduzido (0.5)
resultados_todas_tasks.append(("f2_2_onehot_peso_reduzido", rodar_e_logar(
    "F2-2 OneHot Peso 0.5", {"encoding": "onehot_peso_0.5"},
    [("ord", OrdinalOrdenado(ordem_acabamento)), ("oh_p", OneHotPesoReduzido(['bairro'] + cols_resto_oh, peso=0.5))]
)))

print("\n=== 3. ISSUE #39 — TRATAMENTO SFH (2 variantes) ===")
resultados_todas_tasks.append(("t39_sem_sfh", rodar_e_logar(
    "T39 Sem SFH", {"sfh": "remover"},
    [("sfh", TratamentoSFH('remover'))] + pipe_b0
)))
resultados_todas_tasks.append(("t39_sfh_binarizado", rodar_e_logar(
    "T39 SFH Binarizado", {"sfh": "binarizar"},
    [("sfh", TratamentoSFH('binarizar'))] + pipe_b0
)))

print("\n=== 4. ISSUE #37 — F1-2 FALTANTES (6 variantes) ===")
for est in ['mediana_bairro', 'mediana_bairro_indicador', 'sem_lat_lon', 'area_terreno_tipo', 'media']:
    est_num_final = 'mean' if est == 'media' else 'median'
    resultados_todas_tasks.append((f"f1_2_{est}", rodar_e_logar(
        f"F1-2 {est}", {"faltantes": est},
        [("imp", TratamentoFaltantes(est))] + pipe_b0,
        estrategia_num=est_num_final
    )))

print("\n=== 5. ISSUE #38 — F1-3 OUTLIERS (4 variantes) ===")
# IQR no log (1.5 e 3.0) via filtro_treino (o executor aplica y em R$, então filtro_treino converte para log1p para calcular IQR)
def filtro_iqr_executor(mult):
    def _filtro(X_tr, y_tr):
        y_log = np.log1p(y_tr)
        q1, q3 = y_log.quantile(0.25), y_log.quantile(0.75)
        iqr = q3 - q1
        mask = (y_log >= q1 - mult * iqr) & (y_log <= q3 + mult * iqr)
        return X_tr.loc[mask], y_tr.loc[mask]
    return _filtro

resultados_todas_tasks.append(("f1_3_iqr_log_1_5", rodar_e_logar(
    "F1-3 IQR log 1.5", {"outliers": "iqr_log_1.5"}, pipe_b0, filtro_treino=filtro_iqr_executor(1.5)
)))
resultados_todas_tasks.append(("f1_3_iqr_log_3_0", rodar_e_logar(
    "F1-3 IQR log 3.0", {"outliers": "iqr_log_3.0"}, pipe_b0, filtro_treino=filtro_iqr_executor(3.0)
)))
# Winsorização 1-99 e 5-95 (após imputação para calcular percentis limpos)
resultados_todas_tasks.append(("f1_3_winsor_1_99", rodar_e_logar(
    "F1-3 Winsor 1-99", {"outliers": "winsor_1_99"},
    [("imp", TratamentoFaltantes('media')), ("win", WinsorizadorOutliers(0.01, 0.99))] + pipe_b0
)))
resultados_todas_tasks.append(("f1_3_winsor_5_95", rodar_e_logar(
    "F1-3 Winsor 5-95", {"outliers": "winsor_5_95"},
    [("imp", TratamentoFaltantes('media')), ("win", WinsorizadorOutliers(0.05, 0.95))] + pipe_b0
)))

=== 0. BASELINES B0 (completo e sem_faltantes) ===
[B0 (completo)] MAE=200053.49 | MAPE=0.2716 | R2=0.5668 | cols=131
[B0 (sem_faltantes)] MAE=243210.65 | MAPE=0.3250 | R2=0.6142 | cols=115

=== 1. ENCODING F2-1 (4 variantes) ===
[F2-1 Ordinal] MAE=199588.13 | MAPE=0.2713 | R2=0.5675 | cols=129
[F2-1 Frequency] MAE=173260.12 | MAPE=0.2551 | R2=0.7557 | cols=17
[F2-1 Target Suave (10)] MAE=175574.41 | MAPE=0.2541 | R2=0.7413 | cols=17
[F2-1 Target Forte (100)] MAE=172919.41 | MAPE=0.2541 | R2=0.7754 | cols=17

=== 2. ISSUE #42 — F2-2 VARIANTES EXTRAS (4 variantes) ===
[F2-2 Híbrido] MAE=190722.69 | MAPE=0.2575 | R2=0.5847 | cols=48
[F2-2 Bairro Raros (<30)] MAE=199407.56 | MAPE=0.2686 | R2=0.5670 | cols=99
[F2-2 Sem Bairro] MAE=191079.85 | MAPE=0.2576 | R2=0.5830 | cols=47
[F2-2 OneHot Peso 0.5] MAE=200053.49 | MAPE=0.2716 | R2=0.5668 | cols=131

=== 3. ISSUE #39 — TRATAMENTO SFH (2 variantes) ===
[T39 Sem SFH] MAE=195100.63 | MAPE=0.2667 | R2=0.5676 | cols=130
[T39 SFH Binarizado] MAE=

In [121]:
df_res = pd.read_csv(SAIDA)
cols_vis = ["id", "conjunto", "faltantes", "outliers", "sfh", "encoding", "n_colunas", "mae_media", "mape_media", "rmse_media", "r2_media"]
display(df_res[cols_vis].sort_values("mae_media"))

,id,conjunto,faltantes,outliers,sfh,encoding,n_colunas,mae_media,mape_media,rmse_media,r2_media
5,969fc9a10e98,completo,mediana_global,nenhum,original,target_forte_100,17.0,172919.407803,0.254097,1.002346e+06,0.775389
3,fb7750e1cba5,completo,mediana_global,nenhum,original,frequency,17.0,173260.115628,0.255121,1.036030e+06,0.755744
4,85b2f38818bf,completo,mediana_global,nenhum,original,target_suave_10,17.0,175574.412843,0.254119,1.044874e+06,0.741305
6,bf2b2d2c39d4,completo,mediana_global,nenhum,original,hibrido_te_bairro_oh_resto,48.8,190722.685862,0.257476,1.458232e+06,0.584684
8,7c46464d599d,completo,mediana_global,nenhum,original,onehot_sem_bairro,47.8,191079.852239,0.257625,1.460312e+06,0.583000
10,e24c3983de51,completo,mediana_global,nenhum,remover,onehot,130.7,195100.630746,0.266653,1.490763e+06,0.567649
14,42a25280bb52,completo,sem_lat_lon,nenhum,original,onehot,129.7,195754.349201,0.268452,1.485286e+06,0.571511
12,32ccdf7bd3b5,completo,mediana_bairro,nenhum,original,onehot,131.7,197903.579364,0.268219,1.480660e+06,0.576979
13,f345a4546a65,completo,mediana_bairro_indicador,nenhum,original,onehot,135.6,199289.600439,0.270092,1.480747e+06,0.576922
7,675ec6f1a266,completo,mediana_global,nenhum,original,onehot_bairro_raros_30,99.1,199407.556894,0.268560,1.491697e+06,0.566990
